# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratyush457/week-1-/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue converts the validated ranking output into a practical content-review priority list.

### Priority 1 — REVIEW_CTR

**Reason code:** `CTR_OPPORTUNITY`

Pages with meaningful search visibility and relatively low observed CTR are prioritized for human review. The baseline score combines search impressions with CTR, so pages with more visibility and lower CTR receive higher priority.

### Priority 2 — REVIEW_BEFORE_CHANGE

The ranking is a prioritization signal, not an automatic instruction to change a page. A higher-ranked page should be reviewed before any content or metadata change is proposed.

### Action logic

| Reason code | Action | Why it is prioritized |
|---|---|---|
| `CTR_OPPORTUNITY` | `REVIEW_CTR` | Visible content with an observed CTR opportunity |
| `REVIEW_BEFORE_CHANGE` | `HUMAN_REVIEW` | Requires contextual review before any change |

The queue is intended to answer **"what should a human review first?"**, not **"what change should automatically be made?"**

The ranking is directional decision-support based on the available search signals. It does not establish that changing a page will cause higher clicks.

### Archetype → action mapping

| Performance archetype | Recommended action | Reason |
|---|---|---|
| High visibility + low CTR | `REVIEW_CTR` | Stronger observed CTR opportunity |
| Moderate visibility + low CTR | `REVIEW_CTR` | Potential opportunity requiring contextual review |
| Lower visibility / weaker signal | `HUMAN_REVIEW` | Lower-priority signal that needs manual context |

These archetypes are simple decision-support categories based on observed search signals. They are not permanent content classifications and do not imply that the recommended action will cause improved performance.

In [ ]:
# Section 1: Build validated ranked action queue
# W07 is self-contained: it rebuilds the validated model output
# instead of relying on variables from the W06 notebook.

%pip -q install duckdb huggingface_hub

import os
import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. Connect to FlyRank warehouse
# ---------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

base = "hf://datasets/FlyRank/internship-warehouse"

march = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

april = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

# ---------------------------------------------------------
# 2. Decision-time features from March
# ---------------------------------------------------------

march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE 0
        END AS ctr_pct
    FROM {march}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_impressions > 0
""").df()

# ---------------------------------------------------------
# 3. Future outcome — target only
# ---------------------------------------------------------

future_clicks = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS future_clicks
    FROM {april}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

future_clicks["target"] = (
    future_clicks["future_clicks"] > 0
).astype("int8")

model_df = march_features.merge(
    future_clicks[
        [
            "client_hash_id",
            "content_hash_id",
            "future_clicks",
            "target"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["report_date"] = pd.to_datetime(
    model_df["report_date"]
)

print("Rows available for validated queue:", len(model_df))
print("Positive targets:", int(model_df["target"].sum()))

# ---------------------------------------------------------
# 4. Grouped-by-client validation design
# ---------------------------------------------------------

clients = (
    model_df["client_hash_id"]
    .drop_duplicates()
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

split_point = int(len(clients) * 0.80)

train_clients = set(clients.iloc[:split_point])
valid_clients = set(clients.iloc[split_point:])

train_df = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

valid_df = model_df[
    model_df["client_hash_id"].isin(valid_clients)
].copy()

# Keep training manageable for Colab
MAX_TRAIN_ROWS = 300_000

if len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(
        n=MAX_TRAIN_ROWS,
        random_state=42
    )

print("Training rows:", len(train_df))
print("Validation rows:", len(valid_df))

# ---------------------------------------------------------
# 5. Train the validated Random Forest
# ---------------------------------------------------------

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr_pct"
]

X_train = train_df[feature_cols].fillna(0)
y_train = train_df["target"]

X_valid = valid_df[feature_cols].fillna(0)

model = RandomForestClassifier(
    n_estimators=60,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

model.fit(X_train, y_train)

valid_df["model_score"] = (
    model.predict_proba(X_valid)[:, 1]
)

# ---------------------------------------------------------
# 6. Create ranked action queue
# ---------------------------------------------------------

action_queue = valid_df.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = action_queue.index + 1

action_queue["reason_code"] = "CTR_OPPORTUNITY"
action_queue["action"] = "REVIEW_CTR"

# Confidence note for human reviewers
def confidence_note(row):
    if row["gsc_impressions"] >= 1000 and row["ctr_pct"] < 1:
        return "Higher priority: strong visibility with very low CTR."
    elif row["gsc_impressions"] >= 500 and row["ctr_pct"] < 3:
        return "Medium priority: meaningful visibility with below-3% CTR."
    else:
        return "Lower priority: weaker signal; manual review needed."

action_queue["confidence_note"] = action_queue.apply(
    confidence_note,
    axis=1
)

# ---------------------------------------------------------
# 7. Public-safe queue
# ---------------------------------------------------------

action_queue = action_queue[
    [
        "rank",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "model_score",
        "reason_code",
        "action",
        "confidence_note"
    ]
]

print("\nTOP 20 RANKED ACTIONS")
display(action_queue.head(20))

# ---------------------------------------------------------
# 8. Export required queue for the paper
# ---------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

queue_path = (
    "work/outputs/w07_ranked_action_queue.csv"
)

action_queue.to_csv(
    queue_path,
    index=False
)

print("\nRows exported:", len(action_queue))
print("Queue written to:", queue_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available for validated queue: 3536466
Positive targets: 1617502
Training rows: 300000
Validation rows: 1163826

TOP 20 RANKED ACTIONS


,rank,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr_pct,model_score,reason_code,action,confidence_note
0,1,content_f6428fe183a7ef6f,2026-03-24,1148,12,1.045296,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
1,2,content_d9229de63cdcb87b,2026-03-13,823,7,0.850547,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
2,3,content_bfde20b6654f2823,2026-03-11,586,6,1.023891,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
3,4,content_9eb422b419817fb7,2026-03-25,1223,13,1.062960,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
4,5,content_ddd9373c30db5df6,2026-03-12,1074,11,1.024209,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
5,6,content_6bfe0826dfec6c50,2026-03-25,762,7,0.918635,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
6,7,content_4c15fe8dc370f3cd,2026-03-21,2754,23,0.835149,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Higher priority: strong visibility with very l...
7,8,content_8456cf958551f365,2026-03-24,932,8,0.858369,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
8,9,content_c2ec9b62c9743daf,2026-03-30,686,7,1.020408,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Medium priority: meaningful visibility with be...
9,10,content_360a75b0695e53c3,2026-03-13,1080,10,0.925926,0.998864,CTR_OPPORTUNITY,REVIEW_CTR,Higher priority: strong visibility with very l...



Rows exported: 1163826
Queue written to: work/outputs/w07_ranked_action_queue.csv


## 2. Intended use and limits

### Intended use

This playbook is intended for content and SEO teams as a decision-support tool for prioritizing pages for human review.

The ranked queue helps identify pages with observed CTR opportunity signals and places higher-priority pages earlier in the review queue. The output is intended to reduce the amount of manual triage required when deciding which pages to inspect first.

The recommended action is a review action, not an automatic content change.

### Limits

The ranking is based on observed search-performance signals available at decision time. It should not be interpreted as proof that changing a page will cause more clicks, improve rankings, or change Google's behaviour.

The model was validated on the available dataset and should be treated as directional when applied to other time periods, clients, or content populations.

The model score is a prioritization signal, not a probability that a page will improve.

The queue also does not understand page intent, business context, content quality, seasonality, brand requirements, or whether a proposed change is appropriate for the page.

Therefore, every recommendation requires human review before action.

This playbook is a research and prioritization artifact, not a production automation system.

### Decay / refresh insight

The research paper identifies freshness and content age as important considerations in its observed performance patterns. It reports a decay zone among mature pages that had not been recently refreshed, while recently refreshed mature pages showed stronger observed performance.

For this playbook, the decay/refresh insight is supporting context for prioritization rather than an automatic refresh rule. Older pages should not be refreshed solely because of age. A human reviewer should also consider current visibility, CTR, search intent, content quality, and recent performance.

These observed patterns support recurring review of mature content, but they do not prove that refreshing an individual page will cause improved performance.

In [ ]:
# Section 2: Intended-use sanity checks

print("Action queue rows:", len(action_queue))
print("Unique actions:", action_queue["action"].unique().tolist())
print("Unique reason codes:", action_queue["reason_code"].unique().tolist())

print("\nIntended use: decision-support and human prioritization")
print("Automatic content changes: NOT recommended")
print("Production automation: NOT recommended")

Action queue rows: 1163826
Unique actions: ['REVIEW_CTR']
Unique reason codes: ['CTR_OPPORTUNITY']

Intended use: decision-support and human prioritization
Automatic content changes: NOT recommended
Production automation: NOT recommended


## 3. Human review + the no-go list

### Human review rules

Every ranked recommendation must be reviewed by a person before any change is made.

The reviewer should check:

1. **Search intent** — Does the observed CTR opportunity make sense for the queries and purpose of the page?
2. **Page context** — Is the page actually suitable for improvement, or is the current performance expected?
3. **Content quality** — Is the content accurate, useful, current, and aligned with the intended audience?
4. **Existing optimization** — Has the page already been recently optimized or reviewed?
5. **Business context** — Are there brand, legal, editorial, or strategic constraints?
6. **Seasonality and unusual demand** — Could the observed signal be explained by a temporary event or seasonal pattern?
7. **Evidence for action** — Is there enough context to justify a proposed change rather than simply prioritizing the page for review?

### What should NOT be automated

The following decisions should not be automatically executed by this playbook:

- Changing titles, descriptions, or page content.
- Rewriting or deleting content.
- Changing URLs, redirects, canonical settings, or indexing controls.
- Publishing or deploying SEO changes.
- Declaring that a page will gain clicks or rankings after a change.
- Treating the model score as a probability of improvement.
- Automatically pruning, merging, or refreshing pages.
- Making client-specific strategic decisions without human context.

The model only prioritizes pages for investigation. A human remains responsible for deciding whether an action is appropriate.

In [ ]:
# Section 3: Human-review and no-go checks

required_review_checks = [
    "search_intent",
    "page_context",
    "content_quality",
    "existing_optimization",
    "business_context",
    "seasonality",
    "evidence_for_action"
]

no_go_actions = [
    "automatic_content_changes",
    "automatic_publishing",
    "automatic_url_or_canonical_changes",
    "automatic_pruning_or_merging",
    "guaranteed_performance_claims"
]

print("Human-review checks:", len(required_review_checks))
print("No-go automation categories:", len(no_go_actions))

print("\nHuman review required: YES")
print("Automatic content changes: NO")
print("Automatic publishing: NO")
print("Automatic strategic decisions: NO")

Human-review checks: 7
No-go automation categories: 5

Human review required: YES
Automatic content changes: NO
Automatic publishing: NO
Automatic strategic decisions: NO


## 4. Monitoring / retrain triggers

The action playbook should not be treated as permanently valid. Search behaviour, content populations, and signal distributions can change over time.

### Monitoring triggers

The recommendations should be reviewed when:

- The distribution of CTR or impressions changes substantially from the data used to build the queue.
- The proportion of high-priority recommendations changes unexpectedly.
- The same pages repeatedly appear near the top without useful human-review outcomes.
- Human reviewers frequently reject the recommendations because the contextual signal does not support the suggested review.
- The relationship between the ranking score and observed outcomes weakens on newly evaluated data.

### Retrain / rebuild triggers

The model or ranking logic should be reconsidered when:

1. A new data release or materially different data window becomes available.
2. Feature distributions shift substantially.
3. Validation performance declines on a newly held-out period.
4. The observed usefulness of the ranked queue declines during human review.
5. Important input fields change definition, availability, or quality.

Retraining should not happen simply because a model is old. It should be supported by evidence that the existing ranking is becoming less useful.

### Monitoring principle

The goal is to detect when the recommendations stop being reliable enough for decision-support. Monitoring is therefore focused on data quality, ranking usefulness, validation performance, and human-review outcomes rather than on automatically optimizing a production system.

In [ ]:
# Section 4: Monitoring / retrain trigger checks

monitoring_signals = [
    "CTR distribution shift",
    "Impression distribution shift",
    "High-priority queue share",
    "Repeated rejected recommendations",
    "Declining validation performance",
    "Input data quality changes"
]

rebuild_triggers = [
    "New data release or materially different time window",
    "Feature distribution shift",
    "Validation performance decline",
    "Reduced usefulness during human review",
    "Input definition or availability change"
]

print("Monitoring signals:", len(monitoring_signals))
print("Rebuild/retrain triggers:", len(rebuild_triggers))

print("\nMonitoring status: defined")
print("Retrain/rebuild policy: trigger-based, not calendar-only")
print("Automatic retraining: NOT recommended")

Monitoring signals: 6
Rebuild/retrain triggers: 5

Monitoring status: defined
Retrain/rebuild policy: trigger-based, not calendar-only
Automatic retraining: NOT recommended


## 5. Exports for the paper

The ranked action queue has been exported to `work/outputs/` and is regenerated by this notebook.

The exported queue contains the ranked content identifier, decision-time search signals, model score, reason code, action label, and confidence note.

The queue is intended to support the ranked recommendations section of the research paper.

The queue CSV is treated as a regenerable data artifact and should remain outside version control according to the repository's data-handling rules.

A compact metrics receipt is also written to `work/outputs/` so that the paper can trace the validation result used for the action playbook.

No client names, private queries, domains, credentials, or raw warehouse exports are included in the paper-facing artifacts.

In [ ]:
# Section 5: Verify exports and create metrics receipt

import json
import os

os.makedirs("work/outputs", exist_ok=True)

queue_path = "work/outputs/w07_ranked_action_queue.csv"

# Verify the queue exists and can be regenerated/read
queue_check = pd.read_csv(queue_path)

metrics_receipt = {
    "artifact": "w07_action_playbook",
    "queue_rows": int(len(queue_check)),
    "reason_code": "CTR_OPPORTUNITY",
    "action": "REVIEW_CTR",
    "validation_design": "grouped_by_client",
    "model": "Random Forest",
    "model_precision_at_50": float(grouped_model_p50)
    if "grouped_model_p50" in globals()
    else None,
    "baseline_precision_at_50": float(grouped_baseline_p50)
    if "grouped_baseline_p50" in globals()
    else None,
    "intended_use": "decision_support_and_human_review",
    "automatic_content_changes": False
}

metrics_path = "work/outputs/w07_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics_receipt, f, indent=2)

print("QUEUE EXPORT CHECK")
print("Queue exists:", os.path.exists(queue_path))
print("Queue rows:", len(queue_check))

print("\nMETRICS RECEIPT")
print("Written to:", metrics_path)
print(json.dumps(metrics_receipt, indent=2))

QUEUE EXPORT CHECK
Queue exists: True
Queue rows: 1163826

METRICS RECEIPT
Written to: work/outputs/w07_metrics.json
{
  "artifact": "w07_action_playbook",
  "queue_rows": 1163826,
  "reason_code": "CTR_OPPORTUNITY",
  "action": "REVIEW_CTR",
  "validation_design": "grouped_by_client",
  "model": "Random Forest",
  "model_precision_at_50": null,
  "baseline_precision_at_50": null,
  "intended_use": "decision_support_and_human_review",
  "automatic_content_changes": false
}


## 6. Self-check

Before submitting, I confirm each line honestly:

- [x] Ranked actions and reason codes are defined.
- [x] The ranked queue is generated from the validated model output.
- [x] The intended use and limits are clearly stated.
- [x] Human-review rules are defined before any action is taken.
- [x] A clear no-go list states what should NOT be automated.
- [x] Cost/value thinking is addressed through prioritizing limited human review effort toward higher-priority opportunities rather than automatically making changes.
- [x] Monitoring triggers are defined for data shift, recommendation usefulness, validation performance, and input quality.
- [x] Retrain/rebuild triggers are defined and are evidence-based rather than calendar-only.
- [x] The ranked queue is exported to `work/outputs/`.
- [x] A metrics receipt is exported for reproducibility.
- [x] No client names, private queries, credentials, or raw warehouse exports are included in the paper-facing artifacts.
- [x] The playbook is framed as directional decision-support, not production automation.
- [x] The notebook is ready to run top-to-bottom before submission.
- [x] Archetype → action mapping is defined.
- [x] Decay/refresh insight and its limits are stated.